# Validation of flatcor's three default priors

This notebook presents the validation of [flatcor](https://github.com/dm-k1/flatcor) (3.0, in
development), which offers three default priors for Bayesian linear regression: flat partial *r*,
flat *R* and flat *R*², each a Beta(*a*, 1) prior on *R*² with a uniform direction. The validation
is what someone needs to trust that each prior works within the Bayesian framework. Three things are
validated for every prior, each in two ways (the table in
[`docs/working/REDESIGN.md`](docs/working/REDESIGN.md)):

| | 1. Prior sampling | 2. Posterior estimation | 3. Bayes factors |
|---|---|---|---|
| **a. Computed correctly?** | 1a: the prior draws against the exact prior | 2a: simulation-based calibration | 3a: calibration of the Bayes factor; bridge sampling's error |
| **b. Behaves like a default?** | 1b: invariance | 2b: estimation against a known truth | 3b: the Bayarri et al. (2012) criteria |

flatcor does all three with one Stan model; its Bayes factors estimate each model's marginal
likelihood by bridge sampling. Every check is a standard one, the same for the three priors.

Each study is a folder in `R/2_studies/` (`design.R`, `code.R`), run by the pipeline
[`_targets.R`](_targets.R), that saves its tables to `results/studies/study_<cell>.rds`; the scripts in
`R/3_figures/` make the figures from those files; this notebook only presents them. What each study
tests, expects and why is in [`docs/validation_tests.md`](docs/validation_tests.md). **Results are estimates against
what is expected:** every quantity is shown with its expected value (fixed in advance), its Monte
Carlo standard error (MCSE) and the distance between them in MCSEs, and under each table how many
lie beyond 2 and 3 MCSEs against what chance gives. There is no pass/fail per row; pass/fail is kept
only for tripwires, exact identities that fail only if the code is broken.

**The setting below** chooses what the notebook does:

- `"results"` reads the saved results only (no computing; the default);
- `"cached"` runs the studies first through the pipeline ([`_targets.R`](_targets.R), via
  `run_all.R`), which reruns only what is not done yet or has changed;
- `"fresh"` marks the chosen studies' steps as outdated and reruns them from scratch.

The full validation takes hours (the Bayes factor studies most of it);
[`cluster/README.md`](cluster/README.md) describes a run on a cluster. With `smoke <- TRUE` the
notebook runs or reads the smoke test instead: a corner of each design and a handful of datasets,
to test the code. Its results and figures go to `cache/`, and its numbers mean nothing.

In [ ]:
mode    <- "results"   # "results", "cached" or "fresh"
smoke   <- FALSE       # TRUE: the smoke test instead of the full studies
studies <- c("1a", "1b", "2a", "2b", "3a", "3b")   # the studies to run and show

In [ ]:
## Jupyter starts in the notebook's folder, the project folder, where the scripts run too
source("R/4_notebook/present.R")   # run_studies(), show_study(), show_table()
stopifnot(mode %in% c("results", "cached", "fresh"))
if (mode != "results") {
  runs <- run_studies(studies, fresh = mode == "fresh", smoke = smoke)
  show_table(runs, "This run (exit status 1: the study stopped or a tripwire failed; see its log)")
}
show_table(overview_of(studies, smoke), if (smoke) "Overview (smoke test)" else "Overview")

---

# 1. Prior sampling

## 1a. Does the software draw the prior it claims?

flatcor's Stan model samples *R*, with *R*² ~ Beta(*a*, 1), and a uniform direction, and computes the
coefficients from them. With the data switched off it should draw exactly the prior: the quantiles of
*r*² (one predictor) and of *R*² (three and eight predictors) against those of Beta(*a*, 1), and of
each partial correlation against the prior as a recipe (`draw_coefficients_from_prior()`), with
Monte Carlo SEs that allow for autocorrelated draws.

In [ ]:
show_study("1a", smoke = smoke, tables = list(
  "Quantiles of the prior draws against their expected values" = "quantiles",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "KS distance to the expected distribution (description)" = "ks"
))

## 1b. Is the Bayes factor invariant to the units and coding of the predictors?

Rescaling, shifting or recoding the predictors and the outcome should leave every Bayes factor
unchanged (Bayarri et al., 2012): the change in log Bayes factor, expected 0, against the Monte Carlo
error of bridge sampling.

In [ ]:
show_study("1b", smoke = smoke, tables = list(
  "Mean change in the log Bayes factor, per prior" = "pooled",
  "Change in the log Bayes factor, per dataset" = "invariance",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference"
))

---

# 2. Posterior estimation

## 2a. Is the posterior computed correctly?

Simulation-based calibration (Talts et al., 2018): parameters drawn from each prior, data from them,
the model fitted; if the posterior is right, the rank of each true value among its posterior draws is
uniform. The mean and SD of the scaled ranks and a chi-square over 20 bins against their values under
uniformity, and Stan's divergent transitions.

In [ ]:
show_study("2a", smoke = smoke, tables = list(
  "Ranks of the true values against a uniform distribution" = "ranks",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Divergent transitions of the SBC fits" = "divergences"
))

## 2b. How close are the posterior estimates to a known truth?

On data with known slopes (slope of *x*₁ = 0.2): the bias of *x*₁'s posterior mean, the
coverage of its 95% interval, and the prediction error of the posterior means relative to OLS, for
the three priors, JZS, Zellner–Siow and OLS. The references (bias 0, coverage .95, prediction ratio 1)
are not guaranteed for a Bayesian estimate, so this describes how the priors behave.

In [ ]:
show_study("2b", smoke = smoke, tables = list(
  "Estimates against their reference" = "estimates",
  "Beyond 2 and 3 Monte Carlo SEs of the reference" = "beyond_reference",
  "Stan's divergent transitions" = "divergent"
))

---

# 3. Bayes factors

## 3a. Are the Bayes factors right?

Calibration (Schad et al., 2023): on data from each prior, half from the model without *x*₁, the
share without *x*₁ among datasets with a given BF₁₀ should equal 1 / (1 + BF₁₀); per bin and as a
logistic calibration line (intercept 0, slope 1). And bridge sampling's error: the spread of the
same Bayes factor over five runs with fresh posterior draws against the error bridgesampling reports
(expected ratio 1).

In [ ]:
show_study("3a", smoke = smoke, tables = list(
  "Logistic calibration line per prior and cell" = "calibration_line",
  "Calibration per bin" = "calibration_bins",
  "Bridge sampling's error from repeated runs" = "repeated_runs",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Approximate errors and divergent transitions" = "diagnostics"
))

## 3b. Do the Bayes factors meet the default-prior criteria?

The criteria of Bayarri, Berger, Forte and García-Donato (2012): predictive matching, information
consistency, model-selection consistency for a fixed and a growing number of predictors (*k* at most
90), and intrinsic consistency; invariance is 1b. Where the expectation is a number (predictive
matching, the asymptotic rates) the estimate is shown against it; where it is a direction or a limit
(information consistency, growth with *n*), the table says which.

In [ ]:
show_study("3b", smoke = smoke, tables = list(
  "Predictive matching" = "predictive_matching",
  "Information consistency" = "information_consistency",
  "Rates: slope of the median log Bayes factor against log n" = "rates",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Consistency under the alternative" = "alternative",
  "Intrinsic consistency" = "intrinsic_consistency",
  "Growing k: slope of the median log Bayes factor against log n" = "growing_slopes",
  "Divergent transitions" = "divergences"
))